# Load UniProt official registry

In [2]:
import json
from pathlib import Path

In [7]:
UNIPROT_REGISTRY = Path("database_all_2026_03_02.json")

In [8]:
with UNIPROT_REGISTRY.open() as f:
    registry_data = json.load(f)

In [13]:
print(type(registry_data))
print(registry_data.keys())

<class 'dict'>
dict_keys(['results'])


In [16]:
uniprot_official_name_set = {
    entry["name"].strip().lower() for entry in registry_data["results"] if isinstance(entry, dict) and entry.get("name")
}

print("Official UniProt name count:", len(uniprot_official_name_set))

Official UniProt name count: 185


In [17]:
uniprot_official_set = {
    entry["abbrev"].strip().lower()
    for entry in registry_data["results"]
    if isinstance(entry, dict) and entry.get("abbrev")
}

print("Official UniProt abbrev count:", len(uniprot_official_set))
print("Sample:", sorted(list(uniprot_official_set))[:20])

Official UniProt abbrev count: 185
Sample: ['abcd', 'agora', 'agr', 'allergome', 'alphafolddb', 'antibodypedia', 'antifam', 'arachnoserver', 'araport', 'bgee', 'bindingdb', 'biocyc', 'biogrid', 'biogrid-orcs', 'biomuta', 'bmrb', 'brenda', 'carbonyldb', 'card', 'cazy']


## Load BERDL prefixes.txt

In [18]:
BERDL_PREFIXES = Path("prefixes.txt")

In [19]:
berdl_set = set()

In [20]:
with BERDL_PREFIXES.open() as f:
    for line in f:
        berdl_set.add(line.strip().lower())

print("BERDL idmapping prefixes:", len(berdl_set))

BERDL idmapping prefixes: 103


## Load parquet prefixes 

In [21]:
from pyspark.sql.functions import lower, col

In [24]:
from pyspark.sql import SparkSession

In [25]:
spark = SparkSession.builder.appName("PrefixExploration").getOrCreate()

In [26]:
df = spark.read.parquet("part-00000-0a0d0261-1fee-477d-90d8-1df048058fbf-c000.snappy.parquet")

In [27]:
parquet_set = {row["db"] for row in df.select(lower(col("db")).alias("db")).distinct().collect()}

print("Parquet prefixes:", len(parquet_set))

Parquet prefixes: 82


In [30]:
## Prefixes in parquet but not in UniProt official list
parquet_not_in_uniprot = parquet_set - uniprot_official_set

In [31]:
print(len(parquet_not_in_uniprot))
print(sorted(list(parquet_not_in_uniprot)))

3
['ec', 'ncbitaxon', 'uniprot']


### Interpretation

These are not true registry gaps:

- **ec** – Represents EC numbers. 
- **ncbitaxon** – A naming variation of NCBI Taxonomy.
- **uniprot** – UniProt itself is not listed as an external cross-reference database.

Conclusion: No external databases detected


In [34]:
## Prefixes in BERDL idmapping but not in UniProt official list
berdl_not_in_uniprot = berdl_set - uniprot_official_set

In [35]:
print(len(berdl_not_in_uniprot))
print(sorted(list(berdl_not_in_uniprot)))

25
['crc64', 'embl-cds', 'ensembl_pro', 'ensembl_trs', 'ensemblgenome', 'ensemblgenome_pro', 'ensemblgenome_trs', 'gene_name', 'gene_orderedlocusname', 'gene_orfname', 'gene_synonym', 'gi', 'ncbi_taxid', 'nextprot', 'pharmgkb', 'refseq_nt', 'treefam', 'uniparc', 'uniprotkb-id', 'uniref100', 'uniref50', 'uniref90', 'wbparasite_trs_pro', 'wormbase_pro', 'wormbase_trs']


### Classification of Differences

### A. Internal metadata fields
- crc64
- gene_name
- gene_orfname
- gene_orderedlocusname
- gene_synonym
- uniprotkb-id

These are internal annotations and not external databases.


### B. Subtype namespace variants
- ensembl_pro
- ensembl_trs
- ensemblgenome_pro
- wormbase_pro
- wormbase_trs

These represent subtype-specific namespaces and are collapsed to canonical forms (e.g., `ensembl`).


### C. Legacy identifiers
- gi

Deprecated identifier system.


### D. UniProt internal datasets
- uniref100
- uniref50
- uniref90
- uniparc

These are internal UniProt resources rather than external databases.
